In [ ]:
import base64
import json
from pathlib import Path
import google.generativeai as genai
from PIL import Image
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from threading import Lock
import queue
from datetime import datetime
import inflect

# --- CẤU HÌNH ---
genai.configure(api_key='') 
model = genai.GenerativeModel('gemini-2.5-flash')
p = inflect.engine()

# --- RATE LIMITER (UNCHANGED) ---
class RateLimiter:
    def __init__(self, max_requests=1000, time_window=60):
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = queue.Queue()
        self.lock = Lock()
    
    def wait_if_needed(self):
        with self.lock:
            now = datetime.now()
            while not self.requests.empty():
                oldest = self.requests.queue[0]
                if (now - oldest).total_seconds() > self.time_window:
                    self.requests.get()
                else:
                    break
            if self.requests.qsize() >= self.max_requests:
                oldest = self.requests.queue[0]
                wait_time = self.time_window - (now - oldest).total_seconds()
                if wait_time > 0:
                    print(f"⏸ Waiting {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    self.requests = queue.Queue()
            self.requests.put(datetime.now())

rate_limiter = RateLimiter(max_requests=850, time_window=60)
file_lock = Lock()

# --- SMOOTH PROMPT GENERATION FUNCTION ---
def get_detailed_description(image_path, class_name):
    rate_limiter.wait_if_needed()
    try:
        img = Image.open(image_path)
    except:
        return f"single {class_name} ."

    # 1. Handle singular form
    singular_name = p.singular_noun(class_name)
    if not singular_name: singular_name = class_name
    
    # 2. "Visual Definition" Prompt - more natural
    prompt = f"""
    Look at the image and provide the **visual definition** of a single '{singular_name}'.
    
    Task: Describe the intrinsic physical appearance of just **ONE** instance, as if it were cropped out and isolated.
    
    **STRICT RULES:**
    1. **Subject**: Describe ONE item only. Ignore the background or other identical items in the group.
    2. **Format**: Start with 'single {singular_name}'. Use dot-separated phrases.
    3. **Content**: Focus on Shape, Color, Material.

    **Example for 'keyboard key':** 
    BAD (Describing the group): keyboard key . rows of buttons . full keyboard layout . many keys .
    GOOD (Describing the instance): single keyboard key . square shape . black plastic material . white printed letter . smooth surface .

    **Your output for '{singular_name}':**
    """
    
    try:
        response = model.generate_content([prompt, img])
        text = response.text.strip().replace("\n", " ").replace("..", ".")
        # Fallback if model forgets trailing period
        if not text.endswith('.'): text += ' .'
        return text
    except Exception as e:
        if "429" in str(e):
            time.sleep(60)
            return get_detailed_description(image_path, class_name)
        return f"single {singular_name} ."

def save_result(output_file, result_line):
    with file_lock:
        with open(output_file, 'a', encoding='utf-8') as out:
            out.write(result_line + '\n')

def process_single_image(img_name, class_name, image_dir, output_file):
    image_path = image_dir / img_name
    if not image_path.exists():
        result_line = f"{img_name}\tsingle {class_name} ."
    else:
        desc = get_detailed_description(image_path, class_name)
        result_line = f"{img_name}\t{desc}"
        print(f"✓ {img_name}: {desc[:50]}...")
    
    save_result(output_file, result_line)

def process_in_batches(tasks, image_dir, output_file, batch_size=500, max_workers=40):
    # (Keep batch logic unchanged)
    total = len(tasks)
    processed = 0
    for i in range(0, total, batch_size):
        batch = tasks[i:i + batch_size]
        print(f"\n--- BATCH {i//batch_size + 1} ---")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_single_image, img, cls, image_dir, output_file): img for img, cls in batch}
            for _ in as_completed(futures):
                processed += 1
                if processed % 100 == 0: print(f"Done {processed}/{total}")
        time.sleep(2)

# Main execution
def main():
    input_file = 'data/FSC147/ImageClasses_FSC147.txt'
    output_file = 'data/FSC147/ImageClasses_FSC147_detailed_v6_pos.txt'
    image_dir = Path('data/FSC147/images_384_VarV2')
    
    # Read already processed
    processed_images = set()
    if Path(output_file).exists():
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    img_name = line.split('\t')[0]
                    processed_images.add(img_name)
        print(f"Found {len(processed_images)} already processed images")
    
    # Read tasks
    tasks = []
    with open(input_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split('\t')
            img_name = parts[0]
            class_name = parts[1] if len(parts) > 1 else ''
            
            if img_name in processed_images:
                continue
            
            tasks.append((img_name, class_name))
    
    print(f"\nTotal images to process: {len(tasks)}")
    print(f"Rate limit: 950 requests/minute")
    print(f"Expected time: ~{len(tasks)/950:.1f} minutes\n")
    
    # Process with optimal settings
    process_in_batches(
        tasks, 
        image_dir, 
        output_file,
        batch_size=900,      # Process 900 at a time
        max_workers=50       # 50 parallel workers
    )
    
    print(f"\n{'='*60}")
    print(f"✓ DONE! Saved to {output_file}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Found 6146 already processed images

Total images to process: 0
Rate limit: 950 requests/minute
Expected time: ~0.0 minutes


✓ DONE! Saved to data/FSC147/ImageClasses_FSC147_detailed_v6_pos.txt


In [ ]:
import base64
import json
from pathlib import Path
import google.generativeai as genai
from PIL import Image
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from threading import Lock
import queue
from datetime import datetime, timedelta

# Cấu hình Gemini API
genai.configure(api_key='')
model = genai.GenerativeModel('gemini-2.5-flash')

# Rate limiter
class RateLimiter:
    def __init__(self, max_requests=1000, time_window=60):
        """
        max_requests: 1000 requests
        time_window: 60 seconds (1 minute)
        """
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = queue.Queue()
        self.lock = Lock()
    
    def wait_if_needed(self):
        """Wait if we've hit rate limit"""
        with self.lock:
            now = datetime.now()
            
            # Remove old requests outside time window
            while not self.requests.empty():
                oldest = self.requests.queue[0]
                if (now - oldest).total_seconds() > self.time_window:
                    self.requests.get()
                else:
                    break
            
            # Check if we need to wait
            if self.requests.qsize() >= self.max_requests:
                oldest = self.requests.queue[0]
                wait_time = self.time_window - (now - oldest).total_seconds()
                if wait_time > 0:
                    print(f"⏸ Rate limit reached. Waiting {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    # Clear queue after waiting
                    self.requests = queue.Queue()
            
            # Add current request
            self.requests.put(datetime.now())

# Global rate limiter
rate_limiter = RateLimiter(max_requests=800, time_window=30)  # Buffer: 950 instead of 1000
file_lock = Lock()

def get_detailed_description(image_path, class_name):
    """Get description with rate limiting"""
    rate_limiter.wait_if_needed()
    
    img = Image.open(image_path)
    
    # --- PROMPT MỚI CHO NEGATIVE MINING ---
    prompt = f"""
    Analyze the image and identify visual elements that are **NOT** the object '{class_name}'.
    Your goal is to describe the background, environment, and potential distractors.

    Focus strictly on:
    1. Background materials (e.g., wooden floor, concrete wall, grass, sky, table surface).
    2. Environmental conditions (e.g., blurry background, out of focus, shadows, darkness).
    3. Other objects that are NOT '{class_name}' (e.g., human hands, plates, clutter).

    Output format rules:
    - **Do NOT** describe the '{class_name}' itself.
    - Separate every phrase with a dot and a space ' . '.
    - Do NOT write full sentences.
    - End with a dot.
    
    Example input: Image of a cat on a sofa.
    Example output: beige sofa texture . blurry living room . shadows . fabric patterns . white pillow .

    Your output for this image:
    """
    
    try:
        response = model.generate_content([prompt, img])
        return response.text.strip()
    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower():
            print(f"⚠ Rate limit hit despite throttling. Waiting 60s...")
            time.sleep(60)
            return get_detailed_description(image_path, class_name)  # Retry
        raise e

def save_result(output_file, result_line):
    """Thread-safe file writing"""
    with file_lock:
        with open(output_file, 'a', encoding='utf-8') as out:
            out.write(result_line + '\n')

def process_single_image(img_name, class_name, image_dir, output_file):
    image_path = image_dir / img_name
    
    if not image_path.exists():
        result_line = f"{img_name}\t{class_name}"
    else:
        try:
            detailed_desc = get_detailed_description(image_path, class_name)
            result_line = f"{img_name}\t{detailed_desc}"
            print(f"✓ {img_name}")
        except Exception as e:
            print(f"✗ {img_name} Error: {e}")
            result_line = f"{img_name}\t{class_name}"
    
    # Save immediately (thread-safe)
    save_result(output_file, result_line)
    
    return img_name

def process_in_batches(tasks, image_dir, output_file, batch_size=900, max_workers=50):
    """
    Process in batches to maximize throughput
    
    With 1000 req/min and 50 workers:
    - Each worker can fire ~20 requests/min
    - Total: 1000 requests/min
    """
    total = len(tasks)
    processed = 0
    
    for i in range(0, total, batch_size):
        batch = tasks[i:i + batch_size]
        batch_num = i // batch_size + 1
        total_batches = (total + batch_size - 1) // batch_size
        
        print(f"\n{'='*60}")
        print(f"BATCH {batch_num}/{total_batches} - Processing {len(batch)} images")
        print(f"{'='*60}\n")
        
        start_time = time.time()
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(process_single_image, img_name, class_name, image_dir, output_file): img_name 
                for img_name, class_name in batch
            }
            
            for future in as_completed(futures):
                processed += 1
                elapsed = time.time() - start_time
                rate = processed / elapsed if elapsed > 0 else 0
                eta = (total - processed) / rate if rate > 0 else 0
                
                print(f"Progress: {processed}/{total} ({processed/total*100:.1f}%) | "
                      f"Rate: {rate:.1f} req/s | ETA: {eta/60:.1f}min")
        
        batch_time = time.time() - start_time
        print(f"\nBatch {batch_num} completed in {batch_time:.1f}s")
        
        # Small pause between batches
        if i + batch_size < total:
            print("Waiting 5s before next batch...\n")
            time.sleep(5)

# Main execution
def main():
    input_file = 'data/FSC147/ImageClasses_FSC147.txt'
    output_file = 'data/FSC147/ImageClasses_FSC147_detailed_v6_pos.txt'
    image_dir = Path('data/FSC147/images_384_VarV2')
    
    # Read already processed
    processed_images = set()
    if Path(output_file).exists():
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    img_name = line.split('\t')[0]
                    processed_images.add(img_name)
        print(f"Found {len(processed_images)} already processed images")
    
    # Read tasks
    tasks = []
    with open(input_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split('\t')
            img_name = parts[0]
            class_name = parts[1] if len(parts) > 1 else ''
            
            if img_name in processed_images:
                continue
            
            tasks.append((img_name, class_name))
    
    print(f"\nTotal images to process: {len(tasks)}")
    print(f"Rate limit: 950 requests/minute")
    print(f"Expected time: ~{len(tasks)/950:.1f} minutes\n")
    
    # Process with optimal settings
    process_in_batches(
        tasks, 
        image_dir, 
        output_file,
        batch_size=900,      # Process 900 at a time
        max_workers=50       # 50 parallel workers
    )
    
    print(f"\n{'='*60}")
    print(f"✓ DONE! Saved to {output_file}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()


Total images to process: 6146
Rate limit: 950 requests/minute
Expected time: ~6.5 minutes


BATCH 1/7 - Processing 900 images

✓ 60.jpg
Progress: 1/6146 (0.0%) | Rate: 0.2 req/s | ETA: 514.0min
✓ 6.jpg
Progress: 2/6146 (0.0%) | Rate: 0.4 req/s | ETA: 262.7min
✓ 9.jpg
Progress: 3/6146 (0.0%) | Rate: 0.6 req/s | ETA: 179.3min
✓ 26.jpg
Progress: 4/6146 (0.1%) | Rate: 0.7 req/s | ETA: 136.5min
✓ 21.jpg
✓ 43.jpg
Progress: 5/6146 (0.1%) | Rate: 0.9 req/s | ETA: 114.9min
Progress: 6/6146 (0.1%) | Rate: 1.1 req/s | ETA: 95.7min
✓ 39.jpg
✓ 46.jpg
Progress: 7/6146 (0.1%) | Rate: 1.2 req/s | ETA: 83.7min
Progress: 8/6146 (0.1%) | Rate: 1.4 req/s | ETA: 73.2min
✓ 7.jpg
Progress: 9/6146 (0.1%) | Rate: 1.6 req/s | ETA: 65.1min
✓ 49.jpg
Progress: 10/6146 (0.2%) | Rate: 1.7 req/s | ETA: 58.7min
✓ 63.jpg
Progress: 11/6146 (0.2%) | Rate: 1.9 req/s | ETA: 53.8min
✓ 34.jpg
Progress: 12/6146 (0.2%) | Rate: 2.0 req/s | ETA: 50.9min
✓ 52.jpg✓ 2.jpg
✓ 32.jpg

Progress: 13/6146 (0.2%) | Rate: 2.1 req/s | ETA:

KeyboardInterrupt: 

✓ 1960.jpg
✓ 1977.jpg
✓ 1939.jpg
✓ 1981.jpg
✓ 1944.jpg✓ 1976.jpg

✓ 1896.jpg


In [1]:
def check_miss (input_file, output_file):
    """Check which images are missing in output compared to input"""
    input_images = set()
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                img_name = line.split('\t')[0]
                input_images.add(img_name)
    
    output_images = set()
    if Path(output_file).exists():
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    img_name = line.split('\t')[0]
                    output_images.add(img_name)
    
    missing_images = input_images - output_images
    print(f"Total images in input: {len(input_images)}")
    print(f"Total images in output: {len(output_images)}")
    print(f"Missing images: {len(missing_images)}")
    
    return missing_images

pos_v6 =  open('data/FSC147/ImageClasses_FSC147_detailed_v6_pos.txt').read().splitlines()
className = open('data/FSC147/ImageClasses_FSC147.txt').read().splitlines()

In [6]:
class_v6 = set()
for line in pos_v6:
    parts = line.split(None,1)
    if len(parts) >= 2:
        class_v6.add(parts[0])

class_base = set()
for line in className:
    parts = line.split(None,1)
    if len(parts) >= 2:
        class_base.add(parts[0])
missing = class_base - class_v6
print(f"Missing images in v6: {len(missing)}")
for img in missing:
    for line in className:
        if line.startswith(img + '\t'):
            print(line)
            break

Missing images in v6: 2
6884.jpg	cups
6907.jpg	caps
